# Reduzir dimensão: ver e pré-processar

**Para quem está começando.** Este notebook supõe que você já viu o notebook de clustering, da mesma pasta. A ideia é rodar de cima para baixo, célula por célula, e olhar os números.

O que vamos fazer, em ordem:

1. entender para que serve reduzir dimensão;
2. comprimir 64 colunas com PCA e escolher quantas componentes guardar;
3. ver os mesmos dados em duas dimensões com t-SNE;
4. usar a redução como pré-processamento de um classificador.

Roda no Google Colab ou no Jupyter do seu computador. Só precisa de `numpy`, `pandas`, `scikit-learn` e `matplotlib`.


## Para que serve reduzir

O conjunto de dígitos deste notebook tem 64 colunas: um gráfico tem duas. Reduzir dimensão resolve três problemas de uma vez: permite **ver** os dados, **comprime** a informação (menos colunas, menos tempo e menos memória) e pode **remover ruído**, quando as colunas descartadas carregam mais ruído que sinal.

As duas técnicas daqui fazem isso de maneiras opostas, e a diferença importa para saber quando usar cada uma:

- o **PCA** procura as direções em que os dados mais variam, e projeta tudo nelas. É linear, as componentes são combinações das colunas originais (interpretáveis) e o método preserva a estrutura **global** dos dados;
- o **t-SNE** procura uma posição em duas dimensões que mantenha cada ponto perto dos seus vizinhos mais próximos. É não linear, não devolve componentes reutilizáveis e preserva apenas a estrutura **local**. Ele é para ver, não para medir.

A regra que evita o erro mais comum: use PCA para pré-processar e t-SNE para olhar. Distância entre grupos num gráfico de t-SNE não significa nada.


## Passo 0: ambiente e configuração

Nada para instalar além do que já vem no Colab. Os dados são padronizados antes de qualquer coisa, porque o PCA procura direções de maior variância e uma coluna medida em milhares dominaria as outras.


In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
plt.rcParams["figure.figsize"] = (9, 4)

digitos = load_digits()
X = StandardScaler().fit_transform(digitos.data)
y = digitos.target
print(f"✅ dígitos: {X.shape[0]} linhas | {X.shape[1]} colunas | {len(np.unique(y))} classes")

## Passo 1: quantas componentes o PCA precisa

O PCA ordena as direções da maior para a menor variância. A pergunta prática é quantas guardar, e a resposta sai da curva de variância acumulada: quantas componentes são necessárias para explicar, por exemplo, 90% da variação total.


In [ ]:
pca_completo = PCA(random_state=RANDOM_STATE).fit(X)
acumulada = np.cumsum(pca_completo.explained_variance_ratio_)

for alvo in (0.80, 0.90, 0.95):
    n = int(np.searchsorted(acumulada, alvo) + 1)
    print(f"📊 {alvo:.0%} da variância em {n} componentes de {X.shape[1]}")

fig, ax = plt.subplots()
ax.plot(range(1, len(acumulada) + 1), acumulada, marker="o", markersize=3, color="#0F6B68")
ax.axhline(0.90, color="#B45309", linestyle="--", linewidth=1, label="90% da variância")
ax.set_xlabel("número de componentes")
ax.set_ylabel("variância acumulada")
ax.set_title("A curva que decide quantas componentes guardar")
ax.legend()
plt.show()

In [ ]:
X_pca2 = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)
variancia_2d = pca_completo.explained_variance_ratio_[:2].sum()

fig, ax = plt.subplots(figsize=(7, 5))
dispersao = ax.scatter(X_pca2[:, 0], X_pca2[:, 1], c=y, cmap="tab10", s=12)
ax.set_xlabel("componente 1")
ax.set_ylabel("componente 2")
ax.set_title(f"PCA em 2D: {variancia_2d:.1%} da variância, cor = dígito verdadeiro")
fig.colorbar(dispersao, ax=ax, label="dígito")
plt.tight_layout()
plt.show()

### Como ler isso

A curva de variância acumulada responde "quantas colunas preciso para não perder quase nada", e não "quantas colunas separam as classes". São perguntas diferentes, e essa diferença é a lição do desenho em duas dimensões: as classes aparecem agrupadas, mas se sobrepõem bastante.

O motivo é direto: o PCA é não supervisionado. Ele procura as direções de maior **variância**, que é onde os dados se espalham, e não as direções que melhor **separam** as classes. Nos dígitos, boa parte da variância vem de como cada pessoa escreve, não de qual dígito é. Foi por isso também que a padronização veio antes: sem ela, o PCA escolheria as direções dominadas pelas colunas de maior escala.

E a cor do gráfico não participou de nada: os rótulos entraram só para pintar. Se você remover a linha da cor, o PCA produz exatamente o mesmo resultado.


## Passo 2: os mesmos dados com t-SNE

O t-SNE resolve outro problema: em vez de preservar a variância, ele tenta preservar **quem é vizinho de quem**. Cada ponto é posicionado de forma que seus vizinhos mais próximos no espaço original continuem próximos no desenho, e o `perplexity` diz quantos vizinhos importam (algo como o tamanho da vizinhança que ele tenta respeitar).

Aqui ele roda sobre os dados padronizados. Em dado muito largo, o costume é rodar sobre as primeiras componentes do PCA, que é mais rápido e reduz o ruído.


In [ ]:
inicio = time.perf_counter()
X_tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=RANDOM_STATE).fit_transform(X)
print(f"⏱️ t-SNE em {time.perf_counter() - inicio:.1f}s")

fig, ax = plt.subplots(figsize=(7, 5))
dispersao = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap="tab10", s=12)
ax.set_title("t-SNE em 2D: cor = dígito verdadeiro")
fig.colorbar(dispersao, ax=ax, label="dígito")
plt.tight_layout()
plt.show()

### Como ler isso

A diferença em relação ao PCA é visível: aqui as classes formam ilhas bem separadas, porque o critério do t-SNE é vizinhança e não variância. Dois dígitos que se confundem na escrita aparecem como ilhas vizinhas, e é isso que faz do t-SNE a ferramenta padrão para olhar dado largo.

Agora a parte que costuma ser esquecida, e que vale mais que o gráfico bonito. O que os números do t-SNE **não** significam:

- a **distância entre ilhas** não mede dissimilaridade. Duas ilhas afastadas podem ser as classes mais parecidas, porque o método acomoda os pontos como cabe no plano;
- o **tamanho da bolha** e a densidade dentro dela não significam nada. O t-SNE tende a espalhar pontos densos e a comprimir pontos esparsos;
- **eixos não têm interpretação**: não existe "componente 1", nem direção de maior variância. Girar o gráfico dá um desenho igualmente válido;
- rodadas diferentes com **perplexity** diferente dão desenhos diferentes dos mesmos dados. Teste 5, 30 e 100 antes de contar uma história.

Por tudo isso, t-SNE é ferramenta de inspeção. Para alimentar um modelo, use PCA.


## Passo 3: redução como pré-processamento

Se o PCA não separa melhor as classes no desenho, ele serve para alguma coisa prática? Serve, e é o uso mais comum dele: substituir 64 colunas por poucas componentes e medir o efeito na acurácia de um classificador.


In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

linhas = []
for n_componentes in [2, 5, 10, 20, 40, X.shape[1]]:
    X_reduzido = PCA(n_components=n_componentes, random_state=RANDOM_STATE).fit_transform(X)
    modelo = LogisticRegression(max_iter=5000)
    inicio = time.perf_counter()
    acuracia = cross_val_score(modelo, X_reduzido, y, cv=folds, scoring="accuracy")
    linhas.append({"componentes": n_componentes, "acuracia_cv": acuracia.mean(),
                   "desvio": acuracia.std(), "tempo_s": time.perf_counter() - inicio})

tabela_pca = pd.DataFrame(linhas).set_index("componentes")
display(tabela_pca.round(4))

fig, ax = plt.subplots()
ax.plot(tabela_pca.index, tabela_pca["acuracia_cv"], marker="o", color="#0F6B68")
ax.set_xscale("log")
ax.set_xlabel("componentes guardadas (a última é o dado completo)")
ax.set_ylabel("acurácia (validação cruzada)")
ax.set_title("Acurácia do classificador por número de componentes")
plt.show()

### Como ler isso

A acurácia sobe rápido nas primeiras componentes e satura bem antes das 64 originais. Ou seja: as colunas que o PCA descarta carregavam pouca informação útil para classificar. Com umas 20 componentes o resultado já está perto do dado completo, e com 40 praticamente alcança. É esse o argumento do PCA como pré-processamento, e não a separação visual.

O `tempo_s` aqui não ajuda a decidir: são 64 colunas e 1 797 linhas, e nesse tamanho a variação entre execuções engole o ganho. Em dado largo, onde o número de colunas pesa de verdade, é essa coluna que justifica a redução.

Duas coisas para experimentar: use 2 componentes (a acurácia cai bastante, e isso mostra que o desenho bonito do t-SNE não é o que classifica) e compare com o t-SNE alimentando o mesmo classificador. O resultado do t-SNE costuma ser pior e não reprodutível entre execuções, o que fecha o argumento.

## Resumo

Reduzir dimensão serve para ver, para comprimir e para tirar ruído, e as duas técnicas do notebook não são intercambiáveis. O PCA é linear, ordena as direções por variância, devolve componentes reutilizáveis e preserva a estrutura global: é o que serve como pré-processamento. O t-SNE preserva vizinhança local e faz o melhor desenho bidimensional, ao custo de não ter eixos, não medir distância e mudar a cada execução: é o que serve para inspecionar.

O erro comum, que este notebook mediu nas duas pontas: escolher redução pelo desenho. A acurácia de um classificador com PCA segue a curva de variância, não a beleza do gráfico do t-SNE.

**Próximo passo:** usar a representação reduzida como entrada de um agrupamento, no notebook de clustering, que é a ordem em que esses dois passos aparecem num projeto de verdade.

### Referências

**Análise de componentes principais**

Jolliffe, I. T. (2002). *Principal Component Analysis* (2nd ed.). Springer.

**Visualização de vizinhança**

van der Maaten, L., & Hinton, G. (2008). *Visualizing data using t-SNE.* Journal of Machine Learning Research 9, 2579-2605.

```bibtex
@book{jolliffe2002principal,
  title     = {Principal Component Analysis},
  author    = {Jolliffe, Ian T.},
  year      = {2002},
  edition   = {2},
  publisher = {Springer}
}

@article{vandermaaten2008visualizing,
  title   = {Visualizing data using t-SNE},
  author  = {van der Maaten, Laurens and Hinton, Geoffrey},
  journal = {Journal of Machine Learning Research},
  volume  = {9},
  pages   = {2579--2605},
  year    = {2008}
}
```
